In [69]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path.cwd().parent

RAW_PATH = BASE_DIR / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(RAW_PATH)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [70]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

In [71]:
print(df["TotalCharges"].dtype)

str


In [72]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print(df["TotalCharges"].dtype)

float64


In [73]:
missing_values = df.isnull().sum()

print(missing_values[missing_values > 0])

TotalCharges    11
dtype: int64


In [74]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("Missing TotalCharges:", df["TotalCharges"].isnull().sum())

Missing TotalCharges: 0


In [75]:
df = df.drop(columns=["customerID"])

print("Dataset shape:", df.shape)

Dataset shape: (7043, 20)


In [76]:
# Separate target variable

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print(df["Churn"].value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


In [77]:
# Separate X and y

X = df.drop(columns=["Churn"])
y = df["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 19)
y shape: (7043,)


In [78]:
y.isnull().sum()

np.int64(0)

In [79]:
# Identify numerical and categorical columns

numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


/tmp/ipykernel_222244/1878646873.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(


In [80]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 19)
X_test : (1409, 19)
y_train: (5634,)
y_test : (1409,)


In [81]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_columns
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_columns
        )
    ]
)

In [82]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [83]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()

# X_train[numerical_columns] = scaler.fit_transform(
#     X_train[numerical_columns]
# )

# X_test[numerical_columns] = scaler.transform(
#     X_test[numerical_columns]
# )

In [84]:
# from sklearn.preprocessing import OneHotEncoder

# encoder = OneHotEncoder(
#     handle_unknown="ignore",
#     sparse_output=False
# )

# encoded_train = encoder.fit_transform(
#     X_train[categorical_columns]
# )

# encoded_test = encoder.transform(
#     X_test[categorical_columns]
# )

In [85]:
# import pandas as pd

# encoded_train = pd.DataFrame(
#     encoded_train,
#     columns=encoder.get_feature_names_out(categorical_columns),
#     index=X_train.index
# )

# encoded_test = pd.DataFrame(
#     encoded_test,
#     columns=encoder.get_feature_names_out(categorical_columns),
#     index=X_test.index
# )

In [86]:
# X_train = pd.concat(
#     [
#         X_train[numerical_columns],
#         encoded_train
#     ],
#     axis=1
# )

# X_test = pd.concat(
#     [
#         X_test[numerical_columns],
#         encoded_test
#     ],
#     axis=1
# )

In [87]:
print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape :", X_test_processed.shape)

Processed X_train shape: (5634, 45)
Processed X_test shape : (1409, 45)


In [88]:
print("Training target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training target distribution:
Churn
0    4139
1    1495
Name: count, dtype: int64

Testing target distribution:
Churn
0    1035
1     374
Name: count, dtype: int64


In [90]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print(X_train_processed_df.shape)
print(X_test_processed_df.shape)

(5634, 45)
(1409, 45)


In [91]:
X_train_processed_df.head()

,num__SeniorCitizen,num__tenure,num__MonthlyCharges,num__TotalCharges,cat__gender_Female,cat__gender_Male,cat__Partner_No,cat__Partner_Yes,cat__Dependents_No,cat__Dependents_Yes,...,cat__StreamingMovies_Yes,cat__Contract_Month-to-month,cat__Contract_One year,cat__Contract_Two year,cat__PaperlessBilling_No,cat__PaperlessBilling_Yes,cat__PaymentMethod_Bank transfer (automatic),cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check
3738,-0.441773,0.102371,-0.521976,-0.262257,0.0,1.0,1.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3151,-0.441773,-0.711743,0.337478,-0.503635,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4860,-0.441773,-0.793155,-0.809013,-0.749883,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
3867,-0.441773,-0.263980,0.284384,-0.172722,1.0,0.0,0.0,1.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
3810,-0.441773,-1.281624,-0.676279,-0.989374,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [92]:
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

X_train_processed_df.to_csv(
    PROCESSED_DIR / "X_train.csv",
    index=False
)

X_test_processed_df.to_csv(
    PROCESSED_DIR / "X_test.csv",
    index=False
)

y_train.to_csv(
    PROCESSED_DIR / "y_train.csv",
    index=False
)

y_test.to_csv(
    PROCESSED_DIR / "y_test.csv",
    index=False
)

print("Preprocessed datasets saved successfully.")

Preprocessed datasets saved successfully.


In [93]:
import joblib

PIPELINE_PATH = BASE_DIR / "models" / "preprocessor.pkl"

joblib.dump(
    preprocessor,
    PIPELINE_PATH
)

print("Preprocessor saved to:")
print(PIPELINE_PATH)

Preprocessor saved to:
/home/aximsoft/Downloads/Weekend_Task/AI Customer Intelligence Platform /models/preprocessor.pkl


In [94]:
print("===== PHASE 4 VERIFICATION =====")

print("Original rows:", len(df))

print("Training rows:", len(X_train_processed_df))
print("Testing rows :", len(X_test_processed_df))

print("Training features:", X_train_processed_df.shape[1])
print("Testing features :", X_test_processed_df.shape[1])

print("Missing values in X_train:",
      X_train_processed_df.isnull().sum().sum())

print("Missing values in X_test:",
      X_test_processed_df.isnull().sum().sum())

print("Missing values in y_train:",
      y_train.isnull().sum())

print("Missing values in y_test:",
      y_test.isnull().sum())

===== PHASE 4 VERIFICATION =====
Original rows: 7043
Training rows: 5634
Testing rows : 1409
Training features: 45
Testing features : 45
Missing values in X_train: 0
Missing values in X_test: 0
Missing values in y_train: 0
Missing values in y_test: 0
